# 11 · Feature Engineering

| Section | Features |
|---------|----------|
| 2 · Time | `hour_local`, `day_of_week`, `is_weekend`, `is_peak_hour`, `time_block`, `week_number`, `month` |
| 3 · Distance | `total_distance`, `distance_ratio`, `log_total_dist`, `is_long_trip` |
| 4 · Driver aggregates | all-time, 30-day, 7-day stats + behavioural |
| 5 · Encoding | categorical codes + target encoding |

**Leakage note**: driver stats use `.expanding().shift(1)` so each
row only sees that driver's *past* trips. Target encoding uses the
full dataset here; notebook 12 recomputes from train-only.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import entropy as scipy_entropy

CLEAN_PATH    = Path('../data/processed/cleaned.parquet')
FEAT_PATH     = Path('../data/processed/features.parquet')
MANIFEST_PATH = Path('../data/processed/feature_manifest.json')

SLA_MIN          = 45
MIN_DRIVER_TRIPS = 5

TIME_BLOCK_ORDER = [
    'overnight', 'morning', 'lunch',
    'afternoon', 'dinner', 'late_night',
]

---
## 1 · Load

In [ ]:
df = pd.read_parquet(CLEAN_PATH)

for col in [
    'restaurant_offered_timestamp_utc',
    'order_final_state_timestamp_local',
    'eater_request_timestamp_local',
]:
    df[col] = pd.to_datetime(df[col])

print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')

---
## 2 · Time Features

`eater_request_timestamp_local` is already in local time — extract
hour, day, and block directly.

In [ ]:
req = df['eater_request_timestamp_local']

df['hour_local']  = req.dt.hour
df['day_of_week'] = req.dt.dayofweek          # 0=Mon, 6=Sun
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
df['week_number'] = req.dt.isocalendar().week.astype('Int32')
df['month']       = req.dt.month
df['is_peak_hour'] = df['hour_local'].isin(
    set(range(12, 15)) | set(range(19, 24))
).astype(int)

h = df['hour_local']
df['time_block'] = pd.Categorical(
    np.select(
        condlist=[
            (h >= 6)  & (h < 11),
            (h >= 11) & (h < 14),
            (h >= 14) & (h < 18),
            (h >= 18) & (h < 22),
            (h >= 22),
        ],
        choicelist=[
            'morning', 'lunch', 'afternoon', 'dinner', 'late_night',
        ],
        default='overnight',
    ),
    categories=TIME_BLOCK_ORDER,
    ordered=True,
)

print('Time features done.')

---
## 3 · Distance Features

In [ ]:
df['total_distance']  = df['pickup_distance'] + df['dropoff_distance']
df['distance_ratio']  = (
    df['dropoff_distance'] / (df['pickup_distance'] + 0.001)
)
df['is_long_trip']    = (df['total_distance'] > 5).astype(int)
df['log_pickup']      = np.log1p(df['pickup_distance'])
df['log_dropoff']     = np.log1p(df['dropoff_distance'])
df['log_total_dist']  = np.log1p(df['total_distance'])

print('Distance features done.')

---
## 5 · Driver Aggregate Features

Instead of encoding `driver_uuid` (too high cardinality), we compute
numerical summaries of each driver's historical performance.

**Leakage prevention**: expanding window with `.shift(1)` — each
row only sees prior trips by that driver. Rolling windows use
`closed='left'` for the same reason.

Drivers with < `MIN_DRIVER_TRIPS` prior trips are imputed to the
global median and flagged with `driver_is_new = 1`.

In [ ]:
# Chronological order is required for expanding / rolling windows
df = df.sort_values(
    'restaurant_offered_timestamp_utc'
).reset_index(drop=True)

In [ ]:
g = df.groupby('driver_uuid', sort=False)['ATD']

df['driver_mean_atd']   = g.transform(
    lambda x: x.expanding().mean().shift(1)
)
df['driver_median_atd'] = g.transform(
    lambda x: x.expanding().median().shift(1)
)
df['driver_p90_atd']    = g.transform(
    lambda x: x.expanding().quantile(0.90).shift(1)
)
df['driver_std_atd']    = g.transform(
    lambda x: x.expanding().std().shift(1)
).fillna(0)
df['driver_sla_rate']   = (
    df.groupby('driver_uuid', sort=False)['ATD']
    .transform(lambda x: (x <= SLA_MIN).expanding().mean().shift(1))
)
df['driver_trip_count'] = g.transform(
    lambda x: x.expanding().count().shift(1)
).fillna(0).astype(int)
df['driver_log_trips']  = np.log1p(df['driver_trip_count'])

print('All-time driver stats done.')

In [ ]:
# closed='left' excludes the current trip from its own window
df = df.set_index('restaurant_offered_timestamp_utc')

for window, suffix in [('30D', '30d'), ('7D', '7d')]:
    df[f'driver_mean_atd_{suffix}'] = (
        df.groupby('driver_uuid', sort=False)['ATD']
        .transform(
            lambda x: x.rolling(window, closed='left').mean()
        )
    )
    df[f'driver_trip_count_{suffix}'] = (
        df.groupby('driver_uuid', sort=False)['ATD']
        .transform(
            lambda x: x.rolling(window, closed='left').count()
        )
    ).fillna(0).astype(int)

df['driver_sla_rate_30d'] = (
    df.groupby('driver_uuid', sort=False)['ATD']
    .transform(
        lambda x: (x <= SLA_MIN).rolling('30D', closed='left').mean()
    )
)

df = df.reset_index()
print('Rolling driver stats done.')

In [ ]:
# ── Behavioural features (all-time per driver) ────────────────────────────
df['driver_peak_hour_ratio'] = (
    df.groupby('driver_uuid', sort=False)['is_peak_hour']
    .transform('mean')
)
df['driver_weekend_ratio'] = (
    df.groupby('driver_uuid', sort=False)['is_weekend']
    .transform('mean')
)

# Territory entropy: low = zone specialist, high = generalist
def _entropy(s: pd.Series) -> float:
    counts = s.value_counts(normalize=True)
    return float(scipy_entropy(counts))


driver_entropy = (
    df[df['driver_uuid'] != 'UNKNOWN']
    .groupby('driver_uuid')['territory']
    .apply(_entropy)
)
df['driver_territory_entropy'] = (
    df['driver_uuid'].map(driver_entropy).fillna(0.0)
)

print('Behavioural driver features done.')

In [ ]:
# ── Min-trip guard ─────────────────────────────────────────────────────────
DRIVER_COLS = [
    'driver_mean_atd', 'driver_median_atd', 'driver_p90_atd',
    'driver_std_atd', 'driver_sla_rate',
    'driver_mean_atd_30d', 'driver_sla_rate_30d',
    'driver_mean_atd_7d',
]

new_mask = df['driver_trip_count'] < MIN_DRIVER_TRIPS
df['driver_is_new'] = new_mask.astype('int8')

for col in DRIVER_COLS:
    global_med = df.loc[~new_mask, col].median()
    df.loc[new_mask, col] = global_med
    df[col] = df[col].fillna(global_med)

print(
    f'Min-trip guard: {new_mask.sum():,} rows imputed'
    f' ({new_mask.mean()*100:.1f}%)'
)

In [ ]:
# Consistency quadrant: mean ATD vs std ATD per driver
_ds = (
    df[df['driver_uuid'] != 'UNKNOWN']
    .groupby('driver_uuid')[[
        'driver_mean_atd', 'driver_std_atd', 'driver_sla_rate'
    ]].last()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(
    _ds['driver_mean_atd'], _ds['driver_std_atd'],
    alpha=0.15, s=4, color='steelblue',
)
axes[0].axvline(
    _ds['driver_mean_atd'].median(),
    color='red', linestyle='--', linewidth=1,
)
axes[0].axhline(
    _ds['driver_std_atd'].median(),
    color='orange', linestyle='--', linewidth=1,
)
axes[0].set_xlabel('driver_mean_atd (min)')
axes[0].set_ylabel('driver_std_atd (min)')
axes[0].set_title(
    'Driver Consistency Quadrant\n'
    '(bottom-left = fast & consistent)'
)

axes[1].hist(
    _ds['driver_sla_rate'].dropna(),
    bins=40, color='teal', edgecolor='none',
)
axes[1].set_title('Distribution of driver_sla_rate')
axes[1].set_xlabel('Fraction of trips ≤ 45 min')

plt.tight_layout()
plt.show()

---
## 6 · Categorical Encoding + Target Encoding

In [ ]:
CAT_COLS = [
    'territory', 'courier_flow',
    'geo_archetype', 'merchant_surface', 'time_block',
    'region', 'country_name',   # previously unused — added for coverage
]
CAT_COLS = [c for c in CAT_COLS if c in df.columns]

for col in CAT_COLS:
    if df[col].dtype.name != 'category':
        df[col] = df[col].astype('category')
    df[f'{col}_code'] = df[col].cat.codes

# Target encoding (full dataset — recomputed train-only in nb 12)
df['territory_median_atd'] = (
    df.groupby('territory', observed=True)['ATD']
    .transform('median')
)
df['geo_archetype_median_atd'] = (
    df.groupby('geo_archetype', observed=True)['ATD']
    .transform('median')
)

# Interaction target encodings
df['territory_flow_median_atd'] = (
    df.groupby(
        ['territory', 'courier_flow'], observed=True
    )['ATD']
    .transform('median')
)
df['territory_hour_median_atd'] = (
    df.groupby(
        ['territory', 'hour_local'], observed=True
    )['ATD']
    .transform('median')
)

print('Encoding done.')

---
## 7 · Feature Range Check

In [ ]:
ALL_FEATURES = [
    c for c in (
        # time
        ['hour_local', 'day_of_week', 'is_weekend', 'is_peak_hour',
         'week_number', 'month']
        # distance
        + ['pickup_distance', 'dropoff_distance', 'total_distance',
           'distance_ratio', 'log_pickup', 'log_dropoff',
           'log_total_dist', 'is_long_trip']
        # driver stats
        + ['driver_mean_atd', 'driver_median_atd', 'driver_p90_atd',
           'driver_std_atd', 'driver_sla_rate',
           'driver_trip_count', 'driver_log_trips', 'driver_is_new',
           'driver_mean_atd_30d', 'driver_trip_count_30d',
           'driver_sla_rate_30d',
           'driver_mean_atd_7d', 'driver_trip_count_7d',
           'driver_peak_hour_ratio', 'driver_weekend_ratio',
           'driver_territory_entropy']
        # encoding (numeric codes + target-encoded)
        + ['territory_median_atd', 'geo_archetype_median_atd',
           'territory_flow_median_atd', 'territory_hour_median_atd',
           'territory_code', 'courier_flow_code',
           'geo_archetype_code', 'merchant_surface_code',
           'time_block_code', 'region_code', 'country_name_code']
    )
    # Categorical columns excluded: min/max require ordered dtype
    if c in df.columns
]

range_stats = (
    df[ALL_FEATURES]
    .agg(['min', 'max', 'mean', 'std', 'median',
          lambda x: x.isna().sum()])
    .T
    .rename(columns={'<lambda_0>': 'null_count'})
)
range_stats.insert(
    0, 'dtype',
    [str(df[c].dtype) for c in range_stats.index],
)
range_stats['null_%'] = (
    range_stats['null_count'] / len(df) * 100
).round(2)

# Flag suspicious ranges
range_stats['flag'] = ''
range_stats.loc[range_stats['min'] < 0, 'flag'] += 'negative '
range_stats.loc[range_stats['null_%'] > 5, 'flag'] += 'high_null '
range_stats.loc[
    (range_stats['max'] - range_stats['min']) == 0, 'flag'
] += 'zero_variance '

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', len(range_stats))
display(range_stats.sort_values('null_%', ascending=False))

print(f'\nTotal features checked : {len(ALL_FEATURES)}')
flagged = range_stats[range_stats['flag'].str.strip() != '']
if flagged.empty:
    print('No issues detected.')
else:
    print(f'Flagged features ({len(flagged)}):')
    display(flagged[['dtype', 'min', 'max', 'null_%', 'flag']])

---
## 7 · Save

In [ ]:
NUMERIC_FEATURES = [
    'hour_local', 'day_of_week', 'is_weekend', 'is_peak_hour',
    'week_number', 'month',
    'pickup_distance', 'dropoff_distance', 'total_distance',
    'distance_ratio', 'log_pickup', 'log_dropoff', 'log_total_dist',
    'is_long_trip', 'is_distance_missing',
    'driver_mean_atd', 'driver_median_atd', 'driver_p90_atd',
    'driver_std_atd', 'driver_sla_rate',
    'driver_trip_count', 'driver_log_trips', 'driver_is_new',
    'driver_mean_atd_30d', 'driver_trip_count_30d',
    'driver_sla_rate_30d',
    'driver_mean_atd_7d', 'driver_trip_count_7d',
    'driver_peak_hour_ratio', 'driver_weekend_ratio',
    'driver_territory_entropy',
    'territory_median_atd', 'geo_archetype_median_atd',
    'territory_flow_median_atd', 'territory_hour_median_atd',
    'territory_code', 'courier_flow_code',
    'geo_archetype_code', 'merchant_surface_code', 'time_block_code',
    'region_code', 'country_name_code',
]
NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c in df.columns]

CAT_FEATURES = [
    'territory', 'courier_flow',
    'geo_archetype', 'merchant_surface', 'time_block',
    'region', 'country_name',
]
CAT_FEATURES = [c for c in CAT_FEATURES if c in df.columns]

ID_COLS = [
    'workflow_uuid', 'driver_uuid', 'delivery_trip_uuid',
    'restaurant_offered_timestamp_utc',
]

SAVE_COLS = ID_COLS + NUMERIC_FEATURES + CAT_FEATURES + ['ATD']
SAVE_COLS = [c for c in SAVE_COLS if c in df.columns]

df[SAVE_COLS].to_parquet(FEAT_PATH, index=False, engine='pyarrow')
print(f'features.parquet → {df[SAVE_COLS].shape}  ({FEAT_PATH.stat().st_size/1024**2:.1f} MB)')

manifest = {
    'numeric_features': NUMERIC_FEATURES,
    'cat_features': CAT_FEATURES,
    'target': 'ATD',
    'sla_threshold_min': SLA_MIN,
    'min_driver_trips': MIN_DRIVER_TRIPS,
}
with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)
print('feature_manifest.json saved.')